[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/05_cnn/05_cnn_solutions.ipynb)

# 05. CNN — 연습 문제 해설

[05_cnn.ipynb](05_cnn.ipynb) 끝의 연습 문제 2개에 대한 정답 코드와 해설입니다. **먼저 직접 시도해본 뒤** 참고하세요.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q torch torchvision matplotlib

import time
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
print("device:", device)

transform = transforms.Compose([transforms.ToTensor()])
train_ds = datasets.MNIST(root="../../data", train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root="../../data", train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb).argmax(dim=1)
            correct += (pred == yb).sum().item()
            total += yb.size(0)
    return correct / total

def train_model(model, epochs=3, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
    elapsed = time.time() - t0
    test_acc = evaluate(model, test_loader)
    n_params = sum(p.numel() for p in model.parameters())
    return test_acc, elapsed, n_params

## 연습 1. 같은 3 epoch 기준, 04번 MLP vs CNN 정확도 비교

In [ ]:
class MnistMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * 7 * 7, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

torch.manual_seed(0)
mlp = MnistMLP().to(device)
acc_mlp, time_mlp, params_mlp = train_model(mlp, epochs=3)

torch.manual_seed(0)
cnn = SimpleCNN().to(device)
acc_cnn, time_cnn, params_cnn = train_model(cnn, epochs=3)

print(f"{'모델':<8}{'test_acc':>10}{'학습시간(s)':>14}{'파라미터수':>14}")
print(f"{'MLP':<8}{acc_mlp:>10.4f}{time_mlp:>14.1f}{params_mlp:>14,}")
print(f"{'CNN':<8}{acc_cnn:>10.4f}{time_cnn:>14.1f}{params_cnn:>14,}")

**해설**
- 같은 3 epoch, 비슷한 파라미터 규모에서도 CNN이 MLP보다 test accuracy가 더 높게 나오는 경우가 많습니다.
- 이유는 CNN이 "공간적으로 가까운 픽셀은 서로 관련이 있다"는 이미지의 구조적 특성(inductive bias)을 필터/파라미터 공유를 통해 이미 반영하고 있기 때문입니다. MLP는 이미지를 1차원으로 펼치면서 이 구조 정보를 잃어버립니다.
- 반면 CNN은 Convolution 연산 때문에 같은 epoch 기준 학습 시간이 MLP보다 더 걸릴 수 있습니다 — 정확도와 속도는 트레이드오프입니다.

## 연습 2. Conv 채널 수 / 층 수를 늘리면?

In [ ]:
class DeeperCNN(nn.Module):
    """채널 16->32 를 32->64로 늘리고, conv 블록을 하나 더 추가한 버전"""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 28->14
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 14->7
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.ReLU(),                    # 7->7 (추가 블록)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

torch.manual_seed(0)
deeper_cnn = DeeperCNN().to(device)
acc_deep, time_deep, params_deep = train_model(deeper_cnn, epochs=3)

print(f"{'모델':<12}{'test_acc':>10}{'학습시간(s)':>14}{'파라미터수':>14}")
print(f"{'CNN(기본)':<12}{acc_cnn:>10.4f}{time_cnn:>14.1f}{params_cnn:>14,}")
print(f"{'CNN(확장)':<12}{acc_deep:>10.4f}{time_deep:>14.1f}{params_deep:>14,}")

**해설**
- 채널 수(16→32, 32→64)와 층 수를 늘리면 모델이 더 다양하고 복잡한 특징을 표현할 수 있어 test accuracy가 조금 더 오르는 경향이 있습니다.
- 다만 파라미터 수와 학습 시간이 함께 늘어나므로, 정확도 향상 폭이 크지 않다면 "더 큰 모델"이 항상 정답은 아닙니다.
- MNIST처럼 비교적 쉬운 데이터셋에서는 이미 기본 CNN도 99% 근처에 도달하기 때문에 개선 폭이 작게 보일 수 있습니다 — 더 어려운 데이터셋(CIFAR-10 등)에서 이 트레이드오프가 훨씬 뚜렷하게 드러납니다.